In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
from Preprocessing import Preprocessing_Pipeline
import json


In [3]:
# df = pd.read_csv("data/train-2025.csv",low_memory=False)

# # Parse dates
# df["stmt_date"] = pd.to_datetime(df["stmt_date"])
# df["def_date"]  = pd.to_datetime(df["def_date"])

# # Params you can tune
# LAG_MONTHS   = 4   # reporting lag
# HORIZON_MONTHS = 12
# START_INCL, END_INCL = True, True  # window edge inclusivity

# # 1) Availability date = when the statement becomes usable (prevents look-ahead)
# df["avail_date"] = df["stmt_date"] + pd.DateOffset(months=LAG_MONTHS)

# # 2) Prediction window [pred_start, pred_end]
# df["pred_start"] = df["avail_date"]
# df["pred_end"]   = df["avail_date"] + pd.DateOffset(months=HORIZON_MONTHS)

# # 3) Find NEXT default after each statement date
# def get_next_default(group):
#     result = pd.Series(pd.NaT, index=group.index)
#     for idx, row in group.iterrows():
#         future_defaults = group.loc[
#             (group["def_date"].notna()) & 
#             (group["def_date"] > row["stmt_date"]), 
#             "def_date"
#         ]
#         if len(future_defaults) > 0:
#             result[idx] = future_defaults.min()
#     return result

# df["next_def"] = df.groupby("id", group_keys=False).apply(get_next_default)

# # 4) Build 12-month default label with no look-ahead
# y = pd.Series(0, index=df.index, dtype="Int64")

# has_next_def = df["next_def"].notna()

# # 4a) CRITICAL: Default happened BEFORE the statement became available → cannot use (NA)
# default_before_info = has_next_def & (df["next_def"] < df["avail_date"])
# y[default_before_info] = pd.NA

# # 4b) Default within the prediction window → 1
# if START_INCL and END_INCL:
#     in_win = (
#         has_next_def &
#         (df["next_def"] >= df["pred_start"]) &
#         (df["next_def"] <= df["pred_end"])
#     )
# elif START_INCL and (not END_INCL):
#     in_win = (
#         has_next_def &
#         (df["next_def"] >= df["pred_start"]) &
#         (df["next_def"] <  df["pred_end"])
#     )
# elif (not START_INCL) and END_INCL:
#     in_win = (
#         has_next_def &
#         (df["next_def"] >  df["pred_start"]) &
#         (df["next_def"] <= df["pred_end"])
#     )
# else:
#     in_win = (
#         has_next_def &
#         (df["next_def"] >  df["pred_start"]) &
#         (df["next_def"] <  df["pred_end"])
#     )

# y[in_win] = 1

# df["default_12m"] = y

# print(df["default_12m"].value_counts(dropna=False))

The availability date represents when the financial statement information actually became accessible to the bank, which is critical because you must "only use data that you would have access to in real life" as stated in your project instructions. Financial statements for a given fiscal year (fs_year) are typically filed months after the fiscal year ends, creating a reporting lag.​

Using fs_year would create look-ahead bias or temporal leakage—a serious modeling error where future information contaminates training data. For example, a 2010 financial statement (fs_year=2010) might not be available until mid-2011, but if you split by fs_year, you could train on information that wasn't actually available yet.​

Temporal Splitting Best Practices
Time-based splitting ensures past data trains the model and future data tests it, simulating real-world prediction scenarios. In credit risk modeling, this approach is essential because you're predicting future defaults based on information available at a specific point in time. The Basel II framework emphasizes point-in-time (PIT) credit risk assessment, which measures default risk using information available at the decision point.​

Since you're calculating default_on_statement_date based on whether default occurred within 12 months after avail_date, your target variable is temporally anchored to avail_date. Splitting by avail_date maintains temporal consistency between features and labels, preventing the model from learning relationships that wouldn't exist in production.​

Your project instructions also state the holdout sample is "drawn from a future time period" with no defaults after 12/31/2012 in training data, confirming the temporal nature of the validation strategy.

In [6]:
df=pd.read_csv("data/Default_added.csv",low_memory=False)
df=df.dropna(subset=['default_12m'])

train_df = df[df["avail_date"] < "2012-05-01"].copy()

train_df.to_csv("data/train_data.csv",index=False)

train_df.to_csv("data/train_data_full.csv",index=False)

test_df  = df[df["avail_date"] >= "2012-05-01"].copy()

test_df.to_csv("data/test_data_full.csv",index=False)
raw_test_df = test_df.copy()
raw_test_df.drop(columns=['default_12m','avail_date','pred_start','pred_end','next_def'],inplace=True)
raw_test_df.to_csv("data/raw_test_data.csv",index=False)

In [5]:
df

,Unnamed: 0,id,stmt_date,HQ_city,legal_struct,ateco_sector,def_date,fs_year,asst_intang_fixed,asst_tang_fixed,...,roa,roe,wc_net,margin_fin,cf_operations,avail_date,pred_start,pred_end,next_def,default_12m
0,17,520288,2011-12-31,28.0,SRL,14.0,NaN,2011,67537.0,1137566.0,...,-3.81,-28.03,496258.0,-917029.0,-849.0,2012-04-30,2012-04-30,2013-04-30,NaN,0.0
1,18,520288,2008-12-31,28.0,SRL,14.0,NaN,2008,256438.0,1181416.0,...,-2.76,NaN,97952.0,NaN,-3881.0,2009-04-30,2009-04-30,2010-04-30,NaN,0.0
2,19,520288,2009-12-31,28.0,SRL,14.0,NaN,2009,194046.0,1152014.0,...,-2.17,NaN,-210671.0,NaN,32618.0,2010-04-30,2010-04-30,2011-04-30,NaN,0.0
3,21,520288,2012-12-31,28.0,SRL,14.0,NaN,2012,15195.0,1116938.0,...,-12.99,NaN,367892.0,-1094962.0,-168907.0,2013-04-30,2013-04-30,2014-04-30,NaN,0.0
4,22,520288,2007-12-31,28.0,SRL,14.0,NaN,2007,126603.0,1127807.0,...,6.20,52.43,-317007.0,-1184970.0,80039.0,2008-04-30,2008-04-30,2009-04-30,NaN,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1023547,4885791,92001230520,2011-12-31,52.0,SPA,93.0,NaN,2011,1498628.0,248233.0,...,2.16,-43.00,204689.0,-1286111.0,550755.0,2012-04-30,2012-04-30,2013-04-30,NaN,0.0
1023548,4885792,92001230520,2008-12-31,52.0,SPA,93.0,NaN,2008,2465065.0,343015.0,...,12.97,1.01,-472335.0,-2212557.0,945705.0,2009-04-30,2009-04-30,2010-04-30,NaN,0.0
1023549,4885793,92001230520,2007-12-31,52.0,SPA,93.0,NaN,2007,2375606.0,261775.0,...,6.15,0.62,-78424.0,-2033489.0,1166064.0,2008-04-30,2008-04-30,2009-04-30,NaN,0.0
1023550,4885815,94111750108,2010-12-31,10.0,SRL,68.0,NaN,2010,2973.0,1131586.0,...,-0.31,-1.85,148203.0,-1140392.0,-4793.0,2011-04-30,2011-04-30,2012-04-30,NaN,0.0
